In [1]:
#1.读取原始数据
import pandas as pd

path = r"C:\Users\王睿娴\Desktop\kaggle 数据处理\online_retail分析\online_retail_II.csv"
df=pd.read_csv(path,encoding="latin-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
#2.查看行数和列名
print(df.shape)
print(df.columns.tolist())


(1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [3]:
#3.标准流程：打标签
df["is_missing_id"]=df["Customer ID"].isna()
df["is_cancel"]=df["Invoice"].astype(str).str.startswith("C")
df["is_return"]=df["Quantity"]<0
df["is_bad_price"]=df["Price"]<=0

In [4]:
#4.统计标签数量
print("——————————————数据标记统计————————————")
print("总记录行数：",len(df))
print("Customer ID缺失行数：",len(df["is_missing_id"]))
print("取消订单数：",df["is_cancel"].sum())
print("订单退回数（数量<0）：",df["is_return"].sum())
print("异常价格订单数：",df["is_bad_price"].sum())
print("取消单且数量<0 (重合):", ((df['is_cancel']) & (df['is_return'])).sum())

——————————————数据标记统计————————————
总记录行数： 1067371
Customer ID缺失行数： 1067371
取消订单数： 19494
订单退回数（数量<0）： 22950
异常价格订单数： 6207
取消单且数量<0 (重合): 19493


In [5]:
#5.清洗，剔除无效数据
df_clean=df[~df["is_missing_id"]&~df["is_cancel"]&~df["is_return"]&~df["is_bad_price"]].copy()
print("-----清洗结果-------")
print("清洗后记录数：",len(df_clean))
print("保留数据比例：", round(len(df_clean)/len(df), 2))

-----清洗结果-------
清洗后记录数： 805549
保留数据比例： 0.75


In [6]:
#6.增加金额列
df_clean["Total Price"]=df_clean["Quantity"]*df_clean["Price"]
df_clean["Total Price"].head()

0     83.4
1     81.0
2     81.0
3    100.8
4     30.0
Name: Total Price, dtype: float64

In [7]:
#7.日期转换代码
#转换日期类型
df_clean['InvoiceDate']=pd.to_datetime(df_clean["InvoiceDate"],errors="coerce")
#检查有没有转换失败的
print(df_clean['InvoiceDate'].isna().sum())
#提取年份
df_clean['Year']=df_clean['InvoiceDate'].dt.year
#提取月份
df_clean['Month']=df_clean['InvoiceDate'].dt.month
#提取完整日期
df_clean['Date']=df_clean['InvoiceDate'].dt.date
#检查结果
print(df_clean[['Date','Month','Year']].head(10))

0
         Date  Month  Year
0  2009-12-01     12  2009
1  2009-12-01     12  2009
2  2009-12-01     12  2009
3  2009-12-01     12  2009
4  2009-12-01     12  2009
5  2009-12-01     12  2009
6  2009-12-01     12  2009
7  2009-12-01     12  2009
8  2009-12-01     12  2009
9  2009-12-01     12  2009


In [8]:
#8.保存干净数据
df_clean.to_csv("clean_data_online_retail_II.csv",index=False)

In [9]:
#9.去重并保存文件
import pandas as pd
import numpy as np
data=pd.read_csv("clean_data_online_retail_II.csv")
raw_rows=len(data)
#完全重复统计
dup_number=data.duplicated().sum()
#业务去重
data_detail=data.drop_duplicates(subset=["Invoice","StockCode","Customer ID"],keep="first").reset_index(drop=True)
print("原始行数：",raw_rows)
print("完全重复行数：",dup_number)
print(f"业务去重行数：{raw_rows-len(data_detail)}（包含完全重复行数）")
print("去重后行数：",len(data_detail))
data_detail.to_csv("clean_final.csv", index=False)

原始行数： 805549
完全重复行数： 26124
业务去重行数：36684（包含完全重复行数）
去重后行数： 768865


In [10]:
#10.生成清洗报告
report = f"""
========== Day5 数据清洗与去重报告 ==========
原始行数：{raw_rows}
完全重复行数：{dup_number} 行
业务去重删除：{raw_rows - len(data_detail)} 行（包含完全重复，基于 Invoice+StockCode+Customer ID）
去重后行数：{len(data_detail)}

【去重后关键统计】
唯一客户数：{data_detail['Customer ID'].nunique()}
唯一订单数：{data_detail['Invoice'].nunique()}
时间范围：{data_detail['InvoiceDate'].min()} ~ {data_detail['InvoiceDate'].max()}
总销售额：£{data_detail['Total Price'].sum()}

输出文件：clean_final.csv
==========================================
"""
print(report)
with open("cleaning_report.txt", "w", encoding="utf-8") as f:
    f.write(report)


========== Day5 数据清洗与去重报告 ==========
原始行数：805549
完全重复行数：26124 行
业务去重删除：36684 行（包含完全重复，基于 Invoice+StockCode+Customer ID）
去重后行数：768865

【去重后关键统计】
唯一客户数：5878
唯一订单数：36969
时间范围：2009-12-01 07:45:00 ~ 2011-12-09 12:50:00
总销售额：£17287739.586999997

输出文件：clean_final.csv



In [11]:
import pandas as pd
#11.计算R值
df=pd.read_csv("clean_final.csv")
snapshot_date=pd.to_datetime("2011-12-31")
df["Date"]=pd.to_datetime(df["Date"])
recency = df.groupby('Customer ID')['Date'].max().reset_index()
recency.columns = ['Customer ID', 'Last Purchase Date']
recency['R'] = (snapshot_date - recency['Last Purchase Date']).dt.days
recency['R'] = (snapshot_date - recency['Last Purchase Date']).dt.days
# 查看结果
print(recency.head(10))
print(recency['R'].describe())

   Customer ID Last Purchase Date    R
0      12346.0         2011-01-18  347
1      12347.0         2011-12-07   24
2      12348.0         2011-09-25   97
3      12349.0         2011-11-21   40
4      12350.0         2011-02-02  332
5      12351.0         2010-11-29  397
6      12352.0         2011-11-03   58
7      12353.0         2011-05-19  226
8      12354.0         2011-04-21  254
9      12355.0         2011-05-09  236
count    5878.000000
mean      222.866791
std       209.353961
min        22.000000
25%        47.000000
50%       117.000000
75%       401.000000
max       760.000000
Name: R, dtype: float64


In [12]:
#12.计算F值
frequency=df.groupby('Customer ID')['Invoice'].nunique().reset_index()
frequency.columns=['Customer ID','F']
# 查看结果
print(frequency.head(10))
print(frequency['F'].describe())
rf = recency.merge(frequency, on='Customer ID', how='inner')
print(rf.head())

   Customer ID   F
0      12346.0  12
1      12347.0   8
2      12348.0   5
3      12349.0   4
4      12350.0   1
5      12351.0   1
6      12352.0  10
7      12353.0   2
8      12354.0   1
9      12355.0   2
count    5878.000000
mean        6.289384
std        13.009406
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       398.000000
Name: F, dtype: float64
   Customer ID Last Purchase Date    R   F
0      12346.0         2011-01-18  347  12
1      12347.0         2011-12-07   24   8
2      12348.0         2011-09-25   97   5
3      12349.0         2011-11-21   40   4
4      12350.0         2011-02-02  332   1


In [13]:
#13.计算M值
monetary=df.groupby('Customer ID')['Total Price'].sum().reset_index()
monetary.columns=['Customer ID','M']
#查看结果
print(monetary.head(10))
print(monetary['M'].describe())
rfm = rf.merge(monetary, on='Customer ID', how='inner')
print(rfm.shape)
print(rfm.head())

   Customer ID         M
0      12346.0  77556.46
1      12347.0   4921.53
2      12348.0   1817.80
3      12349.0   4428.69
4      12350.0    334.40
5      12351.0    300.93
6      12352.0   2386.04
7      12353.0    406.76
8      12354.0   1079.40
9      12355.0    947.61
count      5878.000000
mean       2941.092138
std       14428.830739
min           2.950000
25%         338.120000
50%         855.225000
75%        2247.170000
max      580987.040000
Name: M, dtype: float64
(5878, 5)
   Customer ID Last Purchase Date    R   F         M
0      12346.0         2011-01-18  347  12  77556.46
1      12347.0         2011-12-07   24   8   4921.53
2      12348.0         2011-09-25   97   5   1817.80
3      12349.0         2011-11-21   40   4   4428.69
4      12350.0         2011-02-02  332   1    334.40


In [14]:
#15. 保存RFM结果到文件
rfm.to_csv('rfm_result.csv', index=False, encoding='utf-8-sig')

print('已保存 rfm_result.csv，共', len(rfm), '行')

已保存 rfm_result.csv，共 5878 行


In [15]:
#16. 用四分位数将rfm分成四档，生成rfm分数
import pandas as pd
rfm=pd.read_csv("rfm_result.csv")
rfm['R_score']=pd.qcut(rfm['R'],q=4,labels=[4,3,2,1])
rfm['F_score']=pd.qcut(rfm['F'].rank(method='first'),q=4,labels=[1,2,3,4])
rfm['M_score']=pd.qcut(rfm['M'],q=4,labels=[1,2,3,4])
rfm["RFM_score"]=rfm['R_score'].astype(str)+rfm['F_score'].astype(str)+rfm['M_score'].astype(str)
#查看结果
print(rfm.head())
print("\n各分层客户数：")
print(rfm['RFM_score'].value_counts().sort_index())

   Customer ID Last Purchase Date    R   F         M R_score F_score M_score  \
0      12346.0         2011-01-18  347  12  77556.46       2       4       4   
1      12347.0         2011-12-07   24   8   4921.53       4       4       4   
2      12348.0         2011-09-25   97   5   1817.80       3       3       3   
3      12349.0         2011-11-21   40   4   4428.69       4       3       4   
4      12350.0         2011-02-02  332   1    334.40       2       1       1   

  RFM_score  
0       244  
1       444  
2       333  
3       434  
4       211  

各分层客户数：
RFM_score
111    570
112    163
113     34
114      7
121    161
      ... 
433    211
434     69
442      8
443    105
444    651
Name: count, Length: 62, dtype: int64


In [16]:
#17.创建高中低流失函数
def segment_rfm_4(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r >= 3 and f >= 3 and m >= 3:
        return '高价值'
    elif r >= 3 and (f <= 2 or m <= 2):
        return '低价值'
    elif r <= 2 and (f >= 3 or m >= 3):
        return '流失预警'  # 可视为流失大类，或单独
    elif r <= 2 and f <= 2 and m <= 2:
        return '流失'
    else:
        return '中价值'

# 若要把“流失预警”并入“流失”：
def segment_final(row):
    seg = segment_rfm_4(row)
    if seg == '流失预警':
        return '流失'
    return seg

# 应用分层（使用严格4类，预警并入流失；若想保留预警可改用 segment_rfm）
rfm['客户分层'] = rfm.apply(segment_final, axis=1)

# 查看各层客户数
print("\n各客户分层人数：")
print(rfm['客户分层'].value_counts())

# 查看各层在 RFM_score 上的分布（验证）
print("\n各分层对应的 RFM_score 分布：")
print(rfm.groupby('客户分层')['RFM_score'].value_counts())


各客户分层人数：
客户分层
流失     2931
高价值    1814
低价值    1133
Name: count, dtype: int64

各分层对应的 RFM_score 分布：
客户分层  RFM_score
低价值   322          141
      311          138
      422          125
      432          109
      323          106
                  ... 
高价值   433          211
      443          105
      343          101
      334           98
      434           69
Name: count, Length: 62, dtype: int64


In [17]:
#18. 统计各营收占比
#各层客总营收
layer_revenue=rfm.groupby('客户分层')['M'].sum()
#营收占比
revenue_pct=(layer_revenue/layer_revenue.sum()).round(4)
print(revenue_pct.head())

客户分层
低价值    0.0522
流失     0.1895
高价值    0.7583
Name: M, dtype: float64


In [18]:
#19. 平均客单价
layer_count= rfm['客户分层'].value_counts()
summary = pd.concat([layer_count, layer_revenue, revenue_pct], axis=1)
summary.columns=["客户数","总营收","营收占比"]
summary["平均客单价"]=(summary['总营收']/summary['客户数']).round(2)
print(summary)

       客户数           总营收    营收占比    平均客单价
客户分层                                     
流失    2931  3.276782e+06  0.1895  1117.97
高价值   1814  1.310924e+07  0.7583  7226.71
低价值   1133  9.017139e+05  0.0522   795.86


In [19]:
#20. 加入country字段
# 假设客户ID列叫 'Customer ID'，如果是 '客户ID' 自己替换一下
cust_country = df.groupby('Customer ID')['Country'].first()
rfm = rfm.merge(cust_country, left_on='Customer ID', right_index=True, how='left')

In [20]:
#21.制作交叉表
# 营收交叉（行=分层，列=国家，带总计）
ct_rev = pd.crosstab(rfm['客户分层'], rfm['Country'], 
                     values=rfm['M'].sum(), aggfunc='sum', margins=True).round(2)

# 客户数交叉
ct_cnt = pd.crosstab(rfm['客户分层'], rfm['Country'], margins=True)

# 客单价交叉（营收/人数）
ct_avg = (ct_rev / ct_cnt).round(2)

print("【营收交叉】\n", ct_rev)
print("\n【客户数交叉】\n", ct_cnt)
print("\n【客单价交叉】\n", ct_avg)

【营收交叉】
 Country     Australia       Austria      Bahrain       Belgium       Brazil  \
客户分层                                                                          
低价值      3.457548e+07           NaN          NaN  6.915096e+07          NaN   
流失       1.383019e+08  1.210142e+08  34575479.17  1.901651e+08  34575479.17   
高价值      6.915096e+07  8.643870e+07          NaN  2.074529e+08          NaN   
All      2.420284e+08  2.074529e+08  34575479.17  4.667690e+08  34575479.17   

Country       Canada  Channel Islands        Cyprus  Czech Republic  \
客户分层                                                                  
低价值              NaN     1.728774e+07  1.728774e+07     17287739.59   
流失       86438697.94     1.555897e+08  8.643870e+07             NaN   
高价值              NaN     5.186322e+07  5.186322e+07             NaN   
All      86438697.94     2.247406e+08  1.555897e+08     17287739.59   

Country       Denmark  ...         Spain        Sweden   Switzerland  \
客户分层              

In [23]:
#22.保存
with pd.ExcelWriter('rfm_1117_作品集.xlsx', engine='openpyxl') as w:
    summary.to_excel(w, sheet_name='分层汇总(昨日)')
    ct_rev.to_excel(w, sheet_name='营收交叉(全量)')
    ct_cnt.to_excel(w, sheet_name='客户数交叉')
    ct_avg.to_excel(w, sheet_name='客单价交叉')
print("已保存 rfm_1117_作品集.xlsx ✅")

rfm.to_csv('rfm_result.csv', index=False, encoding='utf-8-sig')

已保存 rfm_1117_作品集.xlsx ✅
